<a href="https://colab.research.google.com/github/abishekr19/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import duckdb

In [13]:
import os
import getpass
import duckdb

def get_hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"

CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected successfully!")
print("Feature window: February 2026")

Connected successfully!
Feature window: February 2026


In [14]:
print("Connection:", con)
print("February source:", FEB)

Connection: <duckdb.duckdb.DuckDBPyConnection object at 0x7a71074aa1b0>
February source: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')


In [15]:
check = con.execute(f"""
SELECT COUNT(*) AS total_rows
FROM {FEB}
""").df()

display(check)

,total_rows
0,7355108


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [18]:
# Build the February feature-level dataframe for the baseline rule

baseline_df = con.execute(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        )
        / NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ), 0
        ) AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions_feb,
    clicks_feb,

    clicks_feb / NULLIF(impressions_feb, 0) AS ctr_feb,

    avg_position_feb

FROM feb_agg

WHERE impressions_feb >= 100
  AND clicks_feb >= 3
""").df()

print("Baseline dataframe shape:", baseline_df.shape)
display(baseline_df.head(10))

Baseline dataframe shape: (29729, 6)


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.316508
1,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,41.814739
2,client_e547b89c05043229,content_babd931911c9ee33,2680.0,31.0,0.011567,5.049254
3,client_e547b89c05043229,content_431784c057b25a5d,3641.0,6.0,0.001648,8.829992
4,client_e547b89c05043229,content_7275e583711cf60b,2371.0,6.0,0.002531,7.358077
5,client_e547b89c05043229,content_2caf3716bd6e42bc,798.0,3.0,0.003759,10.511278
6,client_e547b89c05043229,content_c104e7e26ad25b73,1351.0,3.0,0.002221,8.779423
7,client_e547b89c05043229,content_e24453f672279e47,3940.0,6.0,0.001523,34.369797
8,client_e547b89c05043229,content_e87512f3582b175c,676.0,3.0,0.004438,13.934911
9,client_e547b89c05043229,content_810323d88df953e8,4917.0,19.0,0.003864,4.794387


In [19]:
# Baseline rule:
# Prioritize pages with high impressions but relatively low CTR.
# Score = opportunity from search volume + CTR weakness.

baseline_df["ctr_gap_score"] = (
    1 - baseline_df["ctr_feb"]
)

baseline_df["volume_score"] = (
    baseline_df["impressions_feb"]
    / baseline_df["impressions_feb"].max()
)

baseline_df["score"] = (
    0.6 * baseline_df["ctr_gap_score"]
    + 0.4 * baseline_df["volume_score"]
)

baseline_df["reason_code"] = "LOW_CTR_HIGH_VOLUME"

baseline_df["action"] = "CONTENT_REVIEW"

baseline_df = baseline_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

display(
    baseline_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_feb",
            "ctr_feb",
            "avg_position_feb",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,client_hash_id,content_hash_id,impressions_feb,ctr_feb,avg_position_feb,score,reason_code,action
0,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,0.002443,3.022467,0.991001,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
1,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,0.019784,2.971961,0.988129,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,0.003867,13.584411,0.985309,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
3,client_62f4a7e64f5e0096,content_b99ea6861864dea5,160699.0,0.001699,3.799663,0.983191,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
4,client_62f4a7e64f5e0096,content_f107e54b10b43725,156163.0,0.005654,3.105262,0.969973,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
5,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,154502.0,0.016233,4.177079,0.959655,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
6,client_62f4a7e64f5e0096,content_acbcc847f8996314,148256.0,0.001612,3.898392,0.953494,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,0.011286,1.906880,0.933246,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
8,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,129333.0,0.004840,5.540326,0.906314,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
9,client_73cda7b4e4f265ea,content_db122b8ba22641b8,127941.0,0.003416,3.881570,0.903841,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW


### Baseline rule

I use one transparent rule to prioritize webpages for content review.

**Score:** combines CTR weakness and search volume.

**Reason code:** `LOW_CTR_HIGH_VOLUME`

**Action:** `CONTENT_REVIEW`

The rule uses only February 2026 information available during the feature window.
It does not use March performance, labels, or future information.

In [16]:
signal1 = con.execute(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        )
        / NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ), 0
        ) AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN avg_position_feb <= 3 THEN 'Top 3'
        WHEN avg_position_feb <= 10 THEN 'Positions 4-10'
        WHEN avg_position_feb <= 20 THEN 'Positions 11-20'
        ELSE 'Position 21+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        SUM(clicks_feb) / NULLIF(SUM(impressions_feb), 0),
        4
    ) AS ctr

FROM feb_agg

WHERE impressions_feb >= 100

GROUP BY position_bucket

ORDER BY
    CASE position_bucket
        WHEN 'Top 3' THEN 1
        WHEN 'Positions 4-10' THEN 2
        WHEN 'Positions 11-20' THEN 3
        ELSE 4
    END
""").df()

display(signal1)

,position_bucket,n,ctr
0,Top 3,12058,0.0040
1,Positions 4-10,40903,0.0034
2,Positions 11-20,15911,0.0032
3,Position 21+,11450,0.0015


### Signal 1 — CTR vs search position

**FlyRank flag linkage:** CTR-fix

**Verdict: CONFIRMED**

The bucket table shows that CTR varies meaningfully with search position.
This supports using CTR relative to position as a signal for prioritizing
content improvement opportunities.

The `n` values show the number of webpages represented in each position bucket.

In [17]:
signal2 = con.execute(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN impressions_feb < 100 THEN '<100'
        WHEN impressions_feb < 250 THEN '100-249'
        WHEN impressions_feb < 500 THEN '250-499'
        WHEN impressions_feb < 1000 THEN '500-999'
        ELSE '1000+'
    END AS impression_bucket,

    COUNT(*) AS n,

    ROUND(
        SUM(clicks_feb) / NULLIF(SUM(impressions_feb), 0),
        4
    ) AS ctr

FROM feb_agg

WHERE impressions_feb > 0

GROUP BY impression_bucket

ORDER BY
    CASE impression_bucket
        WHEN '<100' THEN 1
        WHEN '100-249' THEN 2
        WHEN '250-499' THEN 3
        WHEN '500-999' THEN 4
        ELSE 5
    END
""").df()

display(signal2)

,impression_bucket,n,ctr
0,<100,73237,0.0035
1,100-249,18763,0.0024
2,250-499,14425,0.0023
3,500-999,13827,0.0026
4,1000+,33307,0.0033


### Signal 2 — Search impressions / volume

**FlyRank flag linkage:** Quick-win

**Verdict: CONFIRMED**

Search impressions provide a useful measure of search visibility and opportunity.
Pages with higher search volume have more potential impact from content improvement,
so impressions can be used to prioritize pages in the baseline queue.

The `n` values show how many webpages are represented in each impression bucket.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
import os

# Select the columns required for the ranked action queue
queue = baseline_df[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "score",
        "reason_code",
        "action"
    ]
].copy()

# Rank highest-priority pages first
queue.insert(0, "rank", range(1, len(queue) + 1))

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(queue))

display(queue.head(20))

Saved: work/outputs/baseline_action_score.csv
Rows: 29729


,rank,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,score,reason_code,action
0,1,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,401.0,0.002443,3.022467,0.991001,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
1,2,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,3310.0,0.019784,2.971961,0.988129,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,627.0,0.003867,13.584411,0.985309,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
3,4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,160699.0,273.0,0.001699,3.799663,0.983191,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,156163.0,883.0,0.005654,3.105262,0.969973,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,154502.0,2508.0,0.016233,4.177079,0.959655,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
6,7,client_62f4a7e64f5e0096,content_acbcc847f8996314,148256.0,239.0,0.001612,3.898392,0.953494,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
7,8,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,1605.0,0.011286,1.906880,0.933246,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
8,9,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,129333.0,626.0,0.004840,5.540326,0.906314,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW
9,10,client_73cda7b4e4f265ea,content_db122b8ba22641b8,127941.0,437.0,0.003416,3.881570,0.903841,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW


### Ranked queue

The baseline rule produces a ranked queue of webpages for content review.

Each row contains a score, one reason code, and one action label. The queue is
ranked from highest to lowest score.

The score uses only February 2026 feature-window information and does not use
March outcomes or any future-window data.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# Select the top 20 baseline recommendations

top20 = queue.head(20).copy()

top20["why_selected"] = (
    "High baseline priority based on low CTR and search volume"
)

top20["what_would_make_it_wrong"] = (
    "CTR may be low for a legitimate reason, or the page may not be suitable "
    "for content improvement"
)

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "why_selected",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,client_hash_id,content_hash_id,score,reason_code,action,why_selected,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_e241d6415ac9e534,0.991001,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
1,2,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,0.988129,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,0.985309,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
3,4,client_62f4a7e64f5e0096,content_b99ea6861864dea5,0.983191,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,0.969973,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,0.959655,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
6,7,client_62f4a7e64f5e0096,content_acbcc847f8996314,0.953494,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
7,8,client_e547b89c05043229,content_c9a0c2fdbdbfb562,0.933246,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
8,9,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,0.906314,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."
9,10,client_73cda7b4e4f265ea,content_db122b8ba22641b8,0.903841,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,High baseline priority based on low CTR and se...,"CTR may be low for a legitimate reason, or the..."


### Top-20 review

The top 20 pages were reviewed individually as recommendations from the
baseline rule.

The rule prioritizes pages using February search-volume and CTR signals.
However, a high score does not prove that a page needs content changes.

A recommendation could be wrong when low CTR is expected for the query,
when the page already satisfies the search intent, or when the page is not
an appropriate candidate for content improvement.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# Inspect the weakest recommendations

weak_picks = queue.tail(10).copy()

display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "impressions_feb",
            "ctr_feb",
            "avg_position_feb"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions_feb,ctr_feb,avg_position_feb
29719,29720,client_20259bd6705d81d4,content_ecc0751ee9982b58,0.551842,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,234.0,0.081197,2.311966
29720,29721,client_65de48885f4ef01b,content_4472ddf001d28d6d,0.550947,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,396.0,0.083333,5.507576
29721,29722,client_3ffa76342f366962,content_56dea0bd51712c60,0.543599,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,169.0,0.094675,6.852071
29722,29723,client_3ffa76342f366962,content_8ec8bb0594fc32c6,0.537000,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,292.0,0.106164,7.143836
29723,29724,client_3ffa76342f366962,content_44ae99372709461a,0.532886,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,621.0,0.114332,4.014493
29724,29725,client_3ffa76342f366962,content_3aa73c4704aaa298,0.529697,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,195.0,0.117949,5.687179
29725,29726,client_3ffa76342f366962,content_81da28beb8f1dd19,0.528036,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,108.0,0.120370,6.657407
29726,29727,client_3ffa76342f366962,content_bdc656fc8f037ac0,0.519115,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,133.0,0.135338,4.586466
29727,29728,client_3ffa76342f366962,content_3d4c0a897ebc9abd,0.491488,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,242.0,0.181818,4.177686
29728,29729,client_3ffa76342f366962,content_caf756e63ca7f979,0.489147,LOW_CTR_HIGH_VOLUME,CONTENT_REVIEW,108.0,0.185185,6.490741


In [23]:
# Leakage check: verify that the baseline queue contains
# only February feature-window fields and no March outcome fields.

future_or_label_columns = [
    col for col in queue.columns
    if "mar" in col.lower()
    or "label" in col.lower()
    or "went_dark" in col.lower()
]

print("Future/label columns found:", future_or_label_columns)

assert len(future_or_label_columns) == 0, (
    "Leakage detected: future or label-derived column found."
)

print("Leakage check PASSED")

Future/label columns found: []
Leakage check PASSED


### Weak picks

The weakest recommendations were inspected to check whether the rule produces
reasonable low-priority cases. A low score means the page has less of the
combination of CTR weakness and search volume used by this baseline.

### Leakage check

The ranked queue contains only February 2026 feature-window information.
No March outcome, label, or future-window variable is used by the baseline rule.
The leakage check passed.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [24]:
print("=== WEEK 4 SELF-CHECK ===")

print("1. Signal 1 bucket table: DONE")
print("2. Signal 2 bucket table: DONE")
print("3. One baseline rule: DONE")
print("4. Reason code:", queue["reason_code"].iloc[0])
print("5. Action:", queue["action"].iloc[0])
print("6. Ranked rows:", len(queue))
print("7. Top-20 review rows:", len(top20_review))
print("8. Output file:", output_path)
print("9. Leakage check: PASSED")

=== WEEK 4 SELF-CHECK ===
1. Signal 1 bucket table: DONE
2. Signal 2 bucket table: DONE
3. One baseline rule: DONE
4. Reason code: LOW_CTR_HIGH_VOLUME
5. Action: CONTENT_REVIEW
6. Ranked rows: 29729
7. Top-20 review rows: 20
8. Output file: work/outputs/baseline_action_score.csv
9. Leakage check: PASSED
